# N03 — Polarizers, wave plates, Jones fields and Stokes observables

**Week 2 · Beginner · 35–50 minutes · CPU / no credentials**

**Virtual apparatus:** linearly polarized low-power source, rotation-mounted polarizer, half-wave plate, quarter-wave plate and a power meter.

**Observable:** transmitted optical power (W), and integrated signed Stokes parameters `[S0,S1,S2,S3]`.

**Assumptions:** single-frequency, fully coherent, ideal transverse Jones field, ideal lossless retarders and perfectly absorbing linear polarizer.

### Predict before you execute
For horizontal input polarization and analyzer angle `θ`: **Malus's law** `Pout=Pinput cos²θ`.

A half-wave plate whose axis is at `α` rotates linear polarization by `2α`. At `α=π/8`, the output is +45° linearly polarized. A quarter-wave plate at `α=π/4` produces circular polarization under the package's explicit signed convention.

**Convention:** `S3=2 Im(∫ conj(Ex) Ey dA)`. Avoid naming handedness without a specified viewing direction and time convention.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from photonicslab import Grid, GaussianBeam, LinearPolarizer, HalfWavePlate, QuarterWavePlate, OpticalBench, PowerMeter

source = GaussianBeam(wavelength=632.8e-9, waist=0.35e-3, power=1e-3).emit(
    Grid(nx=128, ny=128, width=4e-3, height=4e-3))
angles = np.linspace(0, np.pi/2, 13)
measured = np.array([PowerMeter().measure(LinearPolarizer(float(a)).apply(source))
                     for a in angles])
predicted = source.power * np.cos(angles)**2
np.testing.assert_allclose(measured, predicted, rtol=1e-12, atol=1e-15)
print(f"Source: {source.power*1e3:.3f} mW; analyzer at 45°: {measured[6]*1e3:.3f} mW")

fig, ax = plt.subplots(figsize=(6,3.5))
ax.plot(np.rad2deg(angles), predicted*1e3, "k-", label="Malus prediction")
ax.plot(np.rad2deg(angles), measured*1e3, "o", label="PhotonicsLab")
ax.set(xlabel="Analyser angle (degrees)", ylabel="Power (mW)", title="Malus's law")
ax.legend()
plt.show()


### First-principles Jones projector

The axis `u(θ)=(cosθ,sinθ)^T` defines the ideal polarizer Jones matrix `J = u u^T`. For a unit Jones input `e=(1,0)^T`, the relative transmitted power is `||J e||²`. This tiny derivation is valuable even when we use a specialist package for advanced optics.


In [ ]:
theta = np.pi/6
u = np.array([np.cos(theta), np.sin(theta)])
jones_projector = np.outer(u,u)
relative_power = np.vdot(jones_projector @ np.array([1.,0.]),
                         jones_projector @ np.array([1.,0.])).real
assert np.isclose(relative_power, np.cos(theta)**2, atol=1e-14)
assert np.isclose(LinearPolarizer(theta).apply(source).power/source.power,
                  relative_power, atol=1e-13)


### Retarders and signed Stokes parameters

A uniform ideal retarder preserves optical power but changes relative polarization phase. Verify both a half-wave plate and a quarter-wave plate using **integrated** Stokes parameters, not a single pixel. In this ideal model `S0=P` and the fully polarized equality `S1²+S2²+S3²=S0²` holds.


In [ ]:
hwp = HalfWavePlate(np.pi/8).apply(source)
qwp = QuarterWavePlate(np.pi/4).apply(source)
np.testing.assert_allclose(hwp.stokes/source.power, [1,0,1,0], atol=1e-13)
np.testing.assert_allclose(qwp.stokes/source.power, [1,0,0,-1], atol=1e-13)
for output in (hwp,qwp):
    assert np.isclose(output.power, source.power, rtol=1e-12)
    assert np.isclose(np.sum(output.stokes[1:]**2), output.stokes[0]**2, rtol=1e-12)

bench = OpticalBench([HalfWavePlate(np.pi/8), LinearPolarizer(np.pi/4)]).run(source)
print(f"Half-wave output Stokes (normalized): {hwp.stokes/source.power}")
print(f"Quarter-wave output Stokes (normalized): {qwp.stokes/source.power}")
print(f"Aligned analyzer output: {bench.output.power*1e3:.4f} mW")
assert np.isclose(bench.output.power, source.power, rtol=1e-12)


### Bench translation, failures and questions

**Apparatus:** laser approved for a teaching lab, optical mounts, calibrated rotation stage, half/quarter-wave plates specified at the laser wavelength, polarizing analyser, power meter and beam dump. Follow laboratory laser-safety training and approved alignment procedures.

**Measurement sequence:** record dark offset and input power; rotate analyser in measured steps; plot `P(θ)`; fit offset/extinction; insert wave plates and compare analyzer curves. Record angle-zero uncertainty, plate wavelength, retardance tolerance and finite extinction ratio.

**Failure challenge:** choose a plate specified for a different wavelength or add realistic finite extinction. Explain why the ideal Jones prediction changes and how to represent nonideal devices. Do **not** fit physical noise by silently changing the theoretical normalization.

**Assessment:** explain the distinction between pure Jones states, partially polarized mixed beams (coherency/Mueller descriptions) and quantum Fock states. A classical Jones field **cannot predict photon-number statistics**.

**Limitations:** no depolarization, dichroism, spatial walk-off, dispersive retardance or quantum photon statistics. A thin waveplate does not simulate propagation through a birefringent volume.

**Read:** Saleh & Teich, *Fundamentals of Photonics*; Mandel & Wolf, *Optical Coherence and Quantum Optics* (coherence and polarization). **Next:** N04 requires *explicit optical path carrier phase* and beam-splitter port conventions before any interferometer model.
